In [51]:
from diffusers import StableDiffusionPipeline
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers.optimization import get_scheduler
from diffusers.utils.import_utils import is_xformers_available
from diffusers.training_utils import EMAModel
from accelerate import Accelerator
import torch
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm import tqdm

In [52]:
# 设置路径
BASE_MODEL_PATH = "./models/diffusions"
DATASET_DIR = "Paint4Poem-Web-famous-subset"
OUTPUT_DIR = "lora_output"

In [53]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [57]:
# 超参数
lr = 1e-4
batch_size = 1
num_epochs = 5
rank = 4
image_size = 512

IMAGE_SIZE = 512
BATCH_SIZE = 1
EPOCHS = 3
LEARNING_RATE = 1e-4
RANK = 4

In [55]:
# 加载模型组件
tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL_PATH, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL_PATH, subfolder="text_encoder", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(BASE_MODEL_PATH, subfolder="vae", torch_dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained(BASE_MODEL_PATH, subfolder="unet", torch_dtype=torch.float16)


In [58]:
# ====== 数据集定义 ======
class ImageTextDataset(Dataset):
    def __init__(self, dataset_dir, tokenizer):
        self.dataset_dir = dataset_dir
        self.tokenizer = tokenizer
        self.image_files = sorted([f for f in os.listdir(dataset_dir) if f.endswith(".png")])
        self.transform = T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        base = os.path.splitext(self.image_files[idx])[0]
        image = Image.open(os.path.join(self.dataset_dir, f"{base}.png")).convert("RGB")
        with open(os.path.join(self.dataset_dir, f"{base}.txt"), "r", encoding="utf-8") as f:
            caption = f.read().strip()
        pixel_values = self.transform(image)
        encoding = self.tokenizer(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt")
        return {
            "pixel_values": pixel_values,
            "input_ids": encoding.input_ids[0]
        }

In [59]:
# ====== 设置训练器 ======
accelerator = Accelerator(mixed_precision="fp16")

dataset = ImageTextDataset(DATASET_DIR, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

ValueError: fp16 mixed precision requires a GPU (not 'mps').

In [60]:
# ====== LoRA 设置 ======
from diffusers.models.attention_processor import LoRAAttnProcessor
unet.set_attn_processor(LoRAAttnProcessor(hidden_size=unet.config.cross_attention_dim, rank=RANK))
optimizer = torch.optim.AdamW(unet.parameters(), lr=LEARNING_RATE)

unet, optimizer, dataloader, text_encoder, vae = accelerator.prepare(
    unet, optimizer, dataloader, text_encoder, vae
)

TypeError: __init__() got an unexpected keyword argument 'hidden_size'